# Network Intrusion Detection — Supervised NNConv Edge Classifier



## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import NNConv

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score,
    fbeta_score, precision_score, recall_score
)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Helper functions — parse GraphML

In [ ]:
def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph."""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)


def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}
        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except Exception:
                    edge_attrs[attr_name] = data_elem.text
        G.add_edge(source, target, **edge_attrs)

    return G


def extract_edge_features(G):
    """Extract edge features as a pandas DataFrame."""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {'source': u, 'target': v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)

## 3. Load graph

In [ ]:
FILE_PATH = "data/0.1M-Stratified-Multi.graphml"

G_full = parse_graphml(FILE_PATH)
print(f"Full graph  — nodes: {G_full.number_of_nodes():,}  edges: {G_full.number_of_edges():,}")

edge_df = extract_edge_features(G_full)
print("\nEdge feature columns:", edge_df.columns.tolist())
print(edge_df.describe())

## 4. Subsample

CHANGED: target_nodes increased from 3000 to 9000 for richer attack coverage.
CHANGED: manual relabelling replaces nx.convert_node_labels_to_integers to
guarantee edge attributes survive the remapping.

In [ ]:
def get_dense_subgraph(G_full, target_nodes=9000):
    # UNCHANGED: BFS from a high-degree seed node
    random.seed(SEED)
    start_node = random.choice([n for n, d in G_full.degree() if d > 5])

    nodes = {start_node}
    queue = [start_node]

    while len(nodes) < target_nodes and queue:
        current = queue.pop(0)
        for neighbor in G_full.neighbors(current):
            if neighbor not in nodes:
                nodes.add(neighbor)
                queue.append(neighbor)
                if len(nodes) >= target_nodes:
                    break

    return G_full.subgraph(nodes).copy()


def relabel_graph_preserve_attrs(G_sub):
    # CHANGED: manual relabelling — nx.convert_node_labels_to_integers can
    # silently drop edge attributes on MultiGraphs depending on NetworkX version
    node_list = list(G_sub.nodes())
    node2idx  = {n: i for i, n in enumerate(node_list)}

    G_int = nx.MultiGraph()
    G_int.add_nodes_from(range(len(node_list)))

    # copy node attributes
    for n in node_list:
        G_int.nodes[node2idx[n]].update(G_sub.nodes[n])

    # copy edge attributes — iterate all parallel edges explicitly
    for u, v, key, data in G_sub.edges(keys=True, data=True):
        G_int.add_edge(node2idx[u], node2idx[v], key=key, **data)

    return G_int, node2idx


# CHANGED: target_nodes=9000
G_sub = get_dense_subgraph(G_full, target_nodes=9000)
G_int, node2idx = relabel_graph_preserve_attrs(G_sub)
node2idx = {n: n for n in G_int.nodes()}
idx2node = {n: n for n in G_int.nodes()}

print(f"Subgraph   — nodes: {G_sub.number_of_nodes():,}  edges: {G_sub.number_of_edges():,}")
print(f"Relabelled — nodes: {G_int.number_of_nodes():,}  edges: {G_int.number_of_edges():,}")

# sanity check attributes survived
sample = list(G_int.edges(data=True))[:2]
print("\nSanity check G_int edge attrs:")
for u, v, d in sample:
    print(f"  ({u}, {v}): {d}")

## 5. Stratified edge split

CHANGED: supervised framing — attacks are the positive class.
CHANGED: 70/15/15 split applied independently to normal and attack edges
so both classes appear in train, val, and test.
REMOVED: semi-supervised logic that dumped all attacks into test only.

In [ ]:
def preview_graph_edges(G, name, n=5):
    edges = list(G.edges(data=True))[:n]
    print(f"--- {name} (first {n} edges) ---")
    for u, v, edata in edges:
        inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
        label = inner.get('ActivityLabel', 'N/A')
        print(f"  ({u}, {v}) -> ActivityLabel: {label}")


# separate edges by label
normal_edges = []
attack_edges = []

for u, v, edata in G_int.edges(data=True):
    inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
    label = inner.get('ActivityLabel', 0)
    if label == 0:
        normal_edges.append((u, v))
    else:
        attack_edges.append((u, v))

print(f"Normal edges: {len(normal_edges):,}")
print(f"Attack edges: {len(attack_edges):,}")
print(f"Attack ratio in full subgraph: {len(attack_edges)/(len(normal_edges)+len(attack_edges)):.2%}")

# CHANGED: stratified split — 70/15/15 applied independently to each class
random.seed(SEED)
random.shuffle(normal_edges)
random.shuffle(attack_edges)

def split_edges(edges, train_frac=0.70, val_frac=0.15):
    n = len(edges)
    train_end = int(train_frac * n)
    val_end   = int((train_frac + val_frac) * n)
    return edges[:train_end], edges[train_end:val_end], edges[val_end:]

train_normal, val_normal, test_normal = split_edges(normal_edges)
train_attack, val_attack, test_attack = split_edges(attack_edges)

# combine for each split
train_edges = train_normal + train_attack
val_edges   = val_normal   + val_attack
test_edges  = test_normal  + test_attack

# labels: 0 = normal, 1 = attack
train_labels = [0] * len(train_normal) + [1] * len(train_attack)
val_labels   = [0] * len(val_normal)   + [1] * len(val_attack)
test_labels  = [0] * len(test_normal)  + [1] * len(test_attack)

print(f"\nTrain — normal: {len(train_normal):,}  attack: {len(train_attack):,}  "
      f"attack ratio: {len(train_attack)/len(train_edges):.2%}")
print(f"Val   — normal: {len(val_normal):,}  attack: {len(val_attack):,}  "
      f"attack ratio: {len(val_attack)/len(val_edges):.2%}")
print(f"Test  — normal: {len(test_normal):,}  attack: {len(test_attack):,}  "
      f"attack ratio: {len(test_attack)/len(test_edges):.2%}")

## 6. Build training graph and node index map

The training graph is built from train_edges only (both normal and attack).
This is the message-passing backbone — the graph the encoder runs on.

In [ ]:
# CHANGED: node2idx now comes from relabel_graph_preserve_attrs (section 4)
# and covers all nodes in G_int. No redefinition needed here.

def to_idx(edges):
    """Convert (u,v) node-ID pairs to integer index pairs.
    Drops edges whose endpoints are not in node2idx.
    """
    return [
        (node2idx[u], node2idx[v])
        for u, v in edges
        if u in node2idx and v in node2idx
    ]


EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']
NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)

# build training graph — contains all train edges (normal + attack)
# used as the message-passing backbone for the encoder
G_train = nx.MultiGraph()
G_train.add_nodes_from(G_int.nodes(data=True))
G_train.add_edges_from((u, v, G_int[u][v]) for u, v in train_edges)

print(f"G_train — nodes: {G_train.number_of_nodes():,}  edges: {G_train.number_of_edges():,}")
preview_graph_edges(G_train, "G_train")

# convert edge lists to integer indices
train_idx = to_idx(train_edges)
val_idx   = to_idx(val_edges)
test_idx  = to_idx(test_edges)

print(f"\ntrain_idx: {len(train_idx):,}")
print(f"val_idx:   {len(val_idx):,}")
print(f"test_idx:  {len(test_idx):,}")

# drop labels for any edges that got filtered by to_idx
# to_idx drops edges outside node2idx — track which survived
def to_idx_with_labels(edges, labels):
    idx, kept_labels = [], []
    for (u, v), lbl in zip(edges, labels):
        if u in node2idx and v in node2idx:
            idx.append((node2idx[u], node2idx[v]))
            kept_labels.append(lbl)
    return idx, kept_labels

train_idx, train_labels = to_idx_with_labels(train_edges, train_labels)
val_idx,   val_labels   = to_idx_with_labels(val_edges,   val_labels)
test_idx,  test_labels  = to_idx_with_labels(test_edges,  test_labels)

print(f"After to_idx filter:")
print(f"  train: {len(train_idx):,}  val: {len(val_idx):,}  test: {len(test_idx):,}")

## 7. Build PyG Data object

Node features: mean/std/min/max aggregation of incident edge features [N, 28]
plus normalized degree [N, 1] = [N, 29] total.
Edge features: raw traffic attributes per edge [E, 7].
CHANGED: no second normalization pass — features are pre-normalized in GraphML.
CHANGED: degree column only is normalized.

In [ ]:
# node feature aggregation from incident edges
node_feat_dict = {n: [] for n in G_train.nodes()}
for u, v, edata in G_train.edges(data=True):
    for key in edata:
        inner = edata[key]
        feats = [float(inner.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
        node_feat_dict[u].append(feats)
        node_feat_dict[v].append(feats)

node_feats = []
for n in G_train.nodes():
    edge_feats = node_feat_dict[n]
    if edge_feats:
        arr = np.array(edge_feats)
        agg = np.concatenate([
            np.mean(arr, axis=0),
            np.std(arr,  axis=0),
            np.min(arr,  axis=0),
            np.max(arr,  axis=0),
        ])
    else:
        agg = np.zeros(NUM_EDGE_FEATS * 4)
    node_feats.append(agg)

degrees    = np.array([G_train.degree(n) for n in G_train.nodes()], dtype=float).reshape(-1, 1)
node_feats = np.array(node_feats)
node_feats = np.concatenate([node_feats, degrees], axis=1)  # [N, 29]

# CHANGED: normalize degree column only (index -1)
# all other columns are pre-normalized in the GraphML source
deg_col = node_feats[:, -1]
node_feats[:, -1] = (deg_col - deg_col.mean()) / (deg_col.std() + 1e-9)

x = torch.tensor(node_feats, dtype=torch.float)

# edge index and edge_attr for message passing (G_train backbone)
train_edge_list = list(G_train.edges())



edge_index = torch.tensor(
    [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
    dtype=torch.long
).t().contiguous()

edge_attr = []
for u, v in train_edge_list:
    raw_data = G_train[u][v][0]
    feats = [float(raw_data.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    edge_attr.append(feats)

edge_attr = torch.tensor(edge_attr, dtype=torch.float)
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

print("Data object:")
print(f"  x:          {data.x.shape}")
print(f"  edge_index: {data.edge_index.shape}")
print(f"  edge_attr:  {data.edge_attr.shape}")

# edge feature tensors for scoring
def build_edge_attr_tensor(idx_list):
    rows = []
    for u, v in idx_list:
        u_orig, v_orig = idx2node[u], idx2node[v]
        if G_int.has_edge(u_orig, v_orig):
            raw = G_int[u_orig][v_orig][0]
            feats = [float(raw.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
        else:
            feats = [0.0] * NUM_EDGE_FEATS
        rows.append(feats)
    return torch.tensor(rows, dtype=torch.float)

train_edges_attr = build_edge_attr_tensor(train_idx)
val_edges_attr   = build_edge_attr_tensor(val_idx)
test_edges_attr  = build_edge_attr_tensor(test_idx)

print(f"  train_edges_attr: {train_edges_attr.shape}")
print(f"  val_edges_attr:   {val_edges_attr.shape}")
print(f"  test_edges_attr:  {test_edges_attr.shape}")

# sanity check
print(f"\nNode feature matrix: {x.shape}")
print(f"Feature means: {x.mean(dim=0)}")
print(f"Feature stds:  {x.std(dim=0)}")

## 8. Model

CHANGED: three NNConv layers instead of two for 3-hop neighborhood aggregation.
Attack nodes had mean degree 365 vs normal mean 52 — deeper message passing
lets the encoder capture more of that topological contrast.

CHANGED: decoder now has two branches:
  - node branch: concatenated node embeddings [z_u, z_v]
  - edge branch: separate MLP projection of edge features
Both branches are merged before the final classification head.
This prevents the node embeddings from drowning out the edge feature signal
when the hidden dimension is large.

In [ ]:
HIDDEN = 128


class GNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden):
        super().__init__()

        # UNCHANGED: conv1 node_in -> hidden
        self.conv1 = NNConv(
            in_channels=node_in,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, node_in * hidden),
                nn.ReLU()
            )
        )
        # ADDED: BatchNorm after conv1
        self.bn1 = nn.BatchNorm1d(hidden)

        # UNCHANGED: conv2 hidden -> hidden
        self.conv2 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )
        # ADDED: BatchNorm after conv2
        self.bn2 = nn.BatchNorm1d(hidden)

        # ADDED: third NNConv layer for 3-hop aggregation
        self.conv3 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )
        # ADDED: BatchNorm after conv3
        self.bn3 = nn.BatchNorm1d(hidden)

        # CHANGED: decoder uses two branches instead of direct concatenation
        # node_branch: projects [z_u, z_v] -> hidden
        # edge_branch: projects edge_attr -> hidden
        # merge:       combines both branches -> 1 logit
        self.node_branch = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.edge_branch = nn.Sequential(
            nn.Linear(edge_in, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        # merge: hidden (node) + hidden//2 (edge) -> classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden + hidden // 2, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, 1)
        )

    def encode(self, x, edge_index, edge_attr):
        # CHANGED: three conv layers with BatchNorm
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = F.relu(self.bn2(self.conv2(x, edge_index, edge_attr)))
        x = self.bn3(self.conv3(x, edge_index, edge_attr))
        return x

    def decode(self, z, edges, edges_attr):
        u = torch.tensor([e[0] for e in edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in edges], dtype=torch.long)

        # CHANGED: two-branch decoder
        node_feats = torch.cat([z[u], z[v]], dim=1)       # [E, 2*hidden]
        node_out   = self.node_branch(node_feats)          # [E, hidden]
        edge_out   = self.edge_branch(edges_attr)          # [E, hidden//2]
        combined   = torch.cat([node_out, edge_out], dim=1) # [E, hidden + hidden//2]
        return self.classifier(combined).squeeze()         # [E]

    def forward(self, x, edge_index, edge_attr, edges, edges_scoring_attr):
        z = self.encode(x, edge_index, edge_attr)
        return self.decode(z, edges, edges_scoring_attr)


model = GNN(
    node_in=x.shape[1],
    edge_in=NUM_EDGE_FEATS,
    hidden=HIDDEN
)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

## 9. Class imbalance weight

ADDED: pos_weight for BCEWithLogitsLoss to handle the normal/attack imbalance.
pos_weight = num_normal_train / num_attack_train tells the loss to penalize
missing an attack proportionally more than missing a normal edge.

In [ ]:
num_train_normal = train_labels.count(0)
num_train_attack = train_labels.count(1)
pos_weight_val   = num_train_normal / max(num_train_attack, 1)

print(f"Train normal: {num_train_normal:,}  Train attack: {num_train_attack:,}")
print(f"pos_weight:   {pos_weight_val:.2f}  "
      f"(loss penalizes missing an attack {pos_weight_val:.1f}x more than missing a normal edge)")

# CHANGED: pos_weight passed to BCEWithLogitsLoss
pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

## 10. Evaluation functions

In [ ]:
def evaluate(model, data, val_idx, val_edges_attr, val_labels):
    # CHANGED: supervised eval — uses real attack labels, not synthetic non-edges
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)
        logits = model.decode(z, val_idx, val_edges_attr)
        scores = torch.sigmoid(logits).cpu().numpy()

    y_true  = np.array(val_labels)
    val_auc = roc_auc_score(y_true, scores)
    val_ap  = average_precision_score(y_true, scores)

    attack_scores = scores[y_true == 1]
    normal_scores = scores[y_true == 0]

    print(f"  Val AUROC: {val_auc:.4f} | Val AUPRC: {val_ap:.4f} "
          f"| Attack score mean: {attack_scores.mean():.4f} "
          f"| Normal score mean: {normal_scores.mean():.4f}")

    return val_auc


def evaluate_test(model, data, test_idx, test_edges_attr, test_labels):
    # CHANGED: supervised test eval — attack label = 1, normal label = 0
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)
        logits = model.decode(z, test_idx, test_edges_attr)
        scores = torch.sigmoid(logits).cpu().numpy()

    y_true = np.array(test_labels)

    auc = roc_auc_score(y_true, scores)
    ap  = average_precision_score(y_true, scores)

    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true, scores)
    f1_vals    = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-9)
    best_thresh = thresholds[np.argmax(f1_vals)]

    y_pred    = (scores >= best_thresh).astype(int)
    precision = precision_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)
    f2        = fbeta_score(y_true, y_pred, beta=2)

    target_recall = 0.90
    valid = recall_vals >= target_recall
    fpr_at_90_recall = 1 - precision_vals[valid][-1] if valid.any() else float('nan')

    print(f"Test AUROC:            {auc:.4f}")
    print(f"Test AUPRC:            {ap:.4f}")
    print(f"Best threshold:        {best_thresh:.4f}")
    print(f"Precision:             {precision:.4f}")
    print(f"Recall:                {recall:.4f}")
    print(f"F1:                    {f1:.4f}")
    print(f"F2 (recall-weighted):  {f2:.4f}")
    print(f"FPR at 90% Recall:     {fpr_at_90_recall:.4f}")

    return auc, ap, f1, f2

## 11. Training

In [ ]:
BEST_AUC        = 0.0
PATIENCE        = 15
NO_IMPROVE_COUNT = 0
BEST_MODEL_PATH = 'best_model.pt'
EPOCHS          = 300
LR             = 0.0001

# CHANGED: pos_weight already baked into criterion (section 9)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# CHANGED: verbose removed — deprecated in PyTorch 2.2
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

# convert labels to tensors once outside the loop
train_labels_tensor = torch.tensor(train_labels, dtype=torch.float)

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    z = model.encode(data.x, data.edge_index, data.edge_attr)

    # CHANGED: single forward pass over all train edges with their true labels
    # no separate pos/neg scoring — attack=1, normal=0
    train_scores = model.decode(z, train_idx, train_edges_attr)
    loss = criterion(train_scores, train_labels_tensor)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if epoch % 10 == 0:
        val_auc = evaluate(model, data, val_idx, val_edges_attr, val_labels)
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        if val_auc > BEST_AUC:
            BEST_AUC         = val_auc
            NO_IMPROVE_COUNT = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  >> New best val AUROC: {BEST_AUC:.4f} — checkpoint saved")
        else:
            NO_IMPROVE_COUNT += 1
            print(f"  >> No improvement for {NO_IMPROVE_COUNT}/{PATIENCE} evals "
                  f"(best AUROC: {BEST_AUC:.4f})")
            if NO_IMPROVE_COUNT >= PATIENCE:
                print(f"  >> Early stopping triggered at epoch {epoch}.")
                break

        scheduler.step(val_auc)

print(f"\nLoading best model (AUROC: {BEST_AUC:.4f}) from '{BEST_MODEL_PATH}'")
model.load_state_dict(torch.load(BEST_MODEL_PATH))

## 12. Final test evaluation

In [ ]:
auc, ap, f1, f2 = evaluate_test(model, data, test_idx, test_edges_attr, test_labels)

## 13. Score distribution plot

In [ ]:
model.eval()
with torch.no_grad():
    z = model.encode(data.x, data.edge_index, data.edge_attr)

    test_normal_idx_list  = [idx for idx, lbl in zip(test_idx, test_labels) if lbl == 0]
    test_attack_idx_list  = [idx for idx, lbl in zip(test_idx, test_labels) if lbl == 1]
    test_normal_attr = torch.stack([test_edges_attr[i] for i, lbl in enumerate(test_labels) if lbl == 0])
    test_attack_attr = torch.stack([test_edges_attr[i] for i, lbl in enumerate(test_labels) if lbl == 1])

    normal_scores = torch.sigmoid(model.decode(z, test_normal_idx_list, test_normal_attr)).cpu().numpy()
    attack_scores = torch.sigmoid(model.decode(z, test_attack_idx_list, test_attack_attr)).cpu().numpy()

print(f"Normal edges — mean: {normal_scores.mean():.4f}  std: {normal_scores.std():.4f}  "
      f"min: {normal_scores.min():.4f}  max: {normal_scores.max():.4f}")
print(f"Attack edges — mean: {attack_scores.mean():.4f}  std: {attack_scores.std():.4f}  "
      f"min: {attack_scores.min():.4f}  max: {attack_scores.max():.4f}")

plt.figure(figsize=(10, 4))
plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal', color='blue', density=True)
plt.hist(attack_scores, bins=50, alpha=0.6, label='Attack', color='red',  density=True)
plt.xlabel('Model Score (sigmoid)')
plt.ylabel('Density')
plt.title('Score Distribution: Normal vs Attack (Supervised)')
plt.legend()
plt.show()